In [ ]:
# MINI‑GPT FROM SCRATCH (PyTorch)

# Implement a GPT‑2‑style language model exactly like the one used in ChatGPT, Copilot, and the brain of your 
# assistant. By tonight, you'll have a working model that generates text character by character, trained on a 
# small corpus, and you'll understand every line of transformers.GPT2LMHeadModel.

In [3]:
# =============================================================================
# SETUP AND CORPUS — turn raw text into integer IDs a model can train on
# =============================================================================
# Same pipeline idea as notebooks 5–6:
#   raw text → tokens → vocab → integer IDs → model math
# Here a "token" is ONE CHARACTER (simplest tokenizer). Later LLMs use BPE/WordPiece.
#
# PyTorch pieces we import:
#   torch              — tensors + autograd
#   torch.nn (nn)      — layers: Linear, Embedding, Module, ...
#   torch.nn.functional (F) — softmax, cross_entropy, ...
#   numpy / matplotlib — optional helpers / plots

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1) Tiny corpus
# ---------------------------------------------------------------------------
# A short dialogue snippet (Shakespeare-flavored). Small on purpose so training
# is fast and you can inspect every character in the vocab.
# Replace with text = open("myfile.txt").read() later for a bigger experiment.

text = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people."""

# ---------------------------------------------------------------------------
# 2) Character-level vocabulary
# ---------------------------------------------------------------------------
# set(text)  → unique characters in the corpus (letters, space, newline, punct)
# sorted(...) → stable order so IDs do not reshuffle between runs
#
# Example (illustrative):
#   chars ≈ ['\n', ' ', ',', ..., 'a', 'b', ...]
#   char_to_idx['B'] = 8     # "B" becomes integer 8
#   idx_to_char[8] = 'B'     # decode back for printing generations
#
# vocab_size = how many distinct symbols the model must score at each step
# (GPT-2's real vocab is ~50k subwords; ours is ~tens of characters.)

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print(f"Vocabulary size: {vocab_size}")
print("Characters:", chars)
# Expect ~30–40 unique chars on this snippet (letters + whitespace + .,:? etc.)
#
# Next cells usually: encode the whole text to a tensor of ints, then cut
# windows like  context[t : t+block] → target = context[t+1 : t+block+1]
# (next-character prediction — the GPT training objective).

Vocabulary size: 35
Characters: ['\n', ' ', ',', '.', ':', '?', 'A', 'B', 'C', 'F', 'M', 'R', 'S', 'Y', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z']


In [5]:
# =============================================================================
# DATASET AND DATALOADER — windows of context → next-character targets
# =============================================================================
# GPT training objective (language modeling):
#   given characters so far, predict the NEXT character at every position.
#
# Picture (block_size=8 for a tiny sketch; we use 32 below):
#
#   raw text snippet:  B  e  f  o  r  e     w  e
#   as IDs in data:    8 14 17 22 25 14  1 30 14   (made-up numbers)
#
#   One training window starting at i=0:
#     X[0] = [B e f o r e   w]     # context the model SEES (length 8)
#     y[0] = [e f o r e   w e]     # same window shifted +1 (what to PREDICT)
#
#   So at seat 0: see "B" → predict "e"
#      at seat 1: see "e" → predict "f"
#      ...
#      at seat 7: see "w" → predict "e"
#
# Causal attention later makes sure seat t only looks at seats ≤ t,
# matching this "predict next given the past" game.
#

# ---------------------------------------------------------------------------
# 1) Encode the whole corpus to a list of integer IDs
# ---------------------------------------------------------------------------
# "Before" → e.g. [idx('B'), idx('e'), idx('f'), ...]
data = [char_to_idx[ch] for ch in text]

# ---------------------------------------------------------------------------
# 2) Cut every overlapping window of length block_size
# ---------------------------------------------------------------------------
# block_size = context length (how far back one example can see).
# Bigger → more context, more compute / memory.
block_size = 32

def create_sequences(data, block_size):
    """Build (X, y) pairs: X is context, y is next-char targets (X shifted by 1)."""
    X, y = [], []
    # i runs while we still have block_size chars for X AND one extra for y's last target
    for i in range(len(data) - block_size):
        X.append(data[i : i + block_size])           # length block_size
        y.append(data[i + 1 : i + block_size + 1])   # shifted one step right
    return torch.tensor(X), torch.tensor(y)

X, y = create_sequences(data, block_size)
print("X shape:", X.shape, "y shape:", y.shape)
# Expect: (num_windows, block_size) for both
#   num_windows ≈ len(data) - block_size
# Example: ~300 chars, block 32 → roughly 268 windows, each of length 32.

# ---------------------------------------------------------------------------
# 3) Batch them with DataLoader
# ---------------------------------------------------------------------------
# TensorDataset: pairs row X[k] with row y[k]
# DataLoader:    hands the training loop mini-batches of those pairs
#   batch_size=16 → each step sees 16 windows
#   shuffle=True  → random order each epoch (less overfitting to text order)
#
# One batch from loader:
#   xb shape (16, 32)  — 16 contexts
#   yb shape (16, 32)  — 16 matching target sequences
batch_size = 16
dataset = torch.utils.data.TensorDataset(X, y)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


X shape: torch.Size([217, 32]) y shape: torch.Size([217, 32])


In [7]:
# =============================================================================
# GPT-2 STYLE TRANSFORMER BLOCK (Pre-LN + causal self-attention)
# =============================================================================
# One block ≈ "one layer of GPT thinking." Stack many of these → MiniGPT.
#
# Compared to notebook 6's encoder block:
#   - NO cross-attention (decoder-only GPT has no separate encoder memory)
#   - attention is CAUSAL: each seat may look left only (not at the future)
#   - Pre-LN layout (modern GPT-2 style): Norm → Sublayer → add residual
#       x = x + attn(LN(x))
#       x = x + mlp(LN(x))
#
# Picture for characters "B e f o" (T=4 seats):
#
#            looks at →  B   e   f   o
#          seat B        ■   ·   ·   ·     · = blocked (future / right)
#          seat e        ■   ■   ·   ·
#          seat f        ■   ■   ■   ·
#          seat o        ■   ■   ■   ■
#
# Seat "f" can use "B e f" to help predict what comes next — never "o".
#

class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with a lower-triangular (causal) mask."""

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads   # split channels across heads
        self.scale = self.head_dim ** -0.5       # 1/sqrt(d_k) — same reason as nb6
        # One big Linear makes Q, K, V together (3 * embed_dim), then we split
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.proj = nn.Linear(embed_dim, embed_dim)  # mix heads back together

    def forward(self, x):
        # x: (B, T, C) = batch, time/seats, channels (embed_dim)
        B, T, C = x.shape

        # Project to Q,K,V then reshape into heads:
        #   (B, T, 3*C) → (B, T, 3, nh, hs) → (3, B, nh, T, hs)
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each (B, num_heads, T, head_dim)

        # Scores: how much each query seat matches each key seat
        att = (q @ k.transpose(-2, -1)) * self.scale   # (B, nh, T, T)

        # Causal mask: keep lower triangle (tril), wipe upper with -inf
        #   before softmax, -inf → weight ≈ 0 after softmax
        mask = torch.tril(torch.ones(T, T)).view(1, 1, T, T).to(x.device)
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)                   # each query row sums to 1

        # Mix values with those weights, then merge heads → (B, T, C)
        y = att @ v
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.proj(y)


class TransformerBlock(nn.Module):
    """One GPT block: Pre-LN causal attention + Pre-LN MLP, both with residuals."""

    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        # Position-wise MLP: expand → GELU → shrink (same idea as encoder FF)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x):
        # Residual sidewalk + attention edit (after LayerNorm)
        x = x + self.attn(self.ln1(x))
        # Residual sidewalk + MLP edit (after LayerNorm)
        x = x + self.ff(self.ln2(x))
        return x
# Easy story: LN steadies the volume → attn mixes past seats → add back original;
#             LN again → MLP rewrites features per seat → add back again.


In [10]:
# =============================================================================
# MiniGPT — full decoder-only language model (tiny GPT-2 skeleton)
# =============================================================================
# End-to-end picture (one forward pass):
#
#   char IDs  →  token embed + position embed  →  N × TransformerBlock
#            →  final LayerNorm  →  Linear head  →  logits over vocab
#
#   For each seat t: logits[t] = scores for "what character comes next?"
#
# Same family as HuggingFace GPT2LMHeadModel — just much smaller dims.
#

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, ff_dim=128, num_layers=3, block_size=32):
        super().__init__()
        # token_embed: each vocab id → a learnable vector of length embed_dim
        #   idx 8 ("B") → some 64-dim arrow meaning "B"
        self.token_embed = nn.Embedding(vocab_size, embed_dim)

        # pos_embed: WHERE in the window (seat 0, 1, ..., block_size-1)
        #   GPT needs position because attention alone has no left/right order.
        #   Learned table (not sin/cos here) — one vector per seat index.
        self.pos_embed = nn.Parameter(torch.zeros(1, block_size, embed_dim))

        # Stack of GPT blocks (causal attn + MLP), run in sequence
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)          # final norm before the head
        self.head = nn.Linear(embed_dim, vocab_size) # hidden → vocab scores (logits)
        self.block_size = block_size
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # Small random start (GPT-2-ish): helps stable early training
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, idx):
        # idx: (B, T) integer character ids, T must be ≤ block_size
        B, T = idx.shape
        assert T <= self.block_size

        tok_emb = self.token_embed(idx)       # (B, T, C) — WHAT the char is
        pos_emb = self.pos_embed[:, :T, :]    # (1, T, C) — WHERE it sits (broadcast over B)
        x = tok_emb + pos_emb                # combine "what" + "where"

        x = self.blocks(x)                   # deep causal transformer stack
        x = self.ln_f(x)
        logits = self.head(x)                # (B, T, vocab_size)
        return logits
        # Training: compare logits to y (next chars) with cross-entropy.

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive sampling: append one char at a time."""
        # idx starts as a prompt, e.g. encode("Fir") → shape (1, 3)
        for _ in range(max_new_tokens):
            # Model can only see block_size seats — keep the most recent window
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)                 # (1, T, vocab)
            # Only the LAST seat predicts the brand-new next character
            logits = logits[:, -1, :] / temperature  # temperature: <1 sharper, >1 wilder
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # sample 1 id from probs
            idx = torch.cat([idx, next_id], dim=1)             # append → longer sequence
        return idx
# Easy story: forward = "score next char at every seat."
#             generate = "take last seat's scores → sample a char → glue it on → repeat."


In [ ]:
# =============================================================================
# TRAINING LOOP — teach MiniGPT to predict the next character
# =============================================================================
# What "learning" means here:
#   For every seat in xb, model outputs scores over vocab.
#   yb holds the TRUE next character at that seat.
#   Cross-entropy punishes wrong/uncertain guesses; AdamW nudges weights
#   to make the true next char more likely next time.
#
# Typical printed run on this tiny corpus (yours may vary slightly):
#   Epoch 0,   loss ≈ 3.07   ← near random (~log(vocab)≈3.5); barely knows alphabet
#   Epoch 20,  loss ≈ 0.14   ← already memorizing / fitting the short text
#   Epoch 40+, loss ≈ 0.08   ← very low: tiny data is easy to overfit
#   (occasional bumps e.g. ~0.14 at epoch 100 can happen with AdamW / shuffle)
#
# Plot: x = epoch, y = average batch loss that epoch.
#   Healthy = curve drops fast then flattens near a small number.
#   Flat-high = not learning. Exploding/NaN = bug / LR too big.
#

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MiniGPT(vocab_size, embed_dim=64, num_heads=4, ff_dim=128, num_layers=3, block_size=block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)  # adaptive LR optimizer
loss_fn = nn.CrossEntropyLoss()  # expects (N, vocab) logits vs (N,) target ids

epochs = 200
losses = []
for epoch in range(epochs):
    model.train()          # dropout/etc. would turn on here (we have none yet)
    epoch_loss = 0
    for xb, yb in loader:
        # xb: (B, T) context ids   yb: (B, T) next-char target ids
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)  # (B, T, vocab)

        # Flatten seats: CE wants 2D logits + 1D targets
        #   (B*T, vocab) vs (B*T,)  — one classification per character seat
        loss = loss_fn(logits.reshape(-1, vocab_size), yb.reshape(-1))

        optimizer.zero_grad()  # clear old grads
        loss.backward()        # autograd through the whole MiniGPT (the hard part NumPy skipped)
        optimizer.step()       # update embed / attn / mlp / head weights
        epoch_loss += loss.item()

    losses.append(epoch_loss / len(loader))  # mean loss over batches this epoch
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, loss: {losses[-1]:.4f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('MiniGPT training (next-character prediction)')
plt.grid(True, alpha=0.3)
plt.show()
# After this: model.generate(...) should spit Shakespeare-ish characters
# from the tiny corpus (often memorized phrases more than true creativity).


Epoch 0, loss: 3.1308
Epoch 20, loss: 0.1294
Epoch 40, loss: 0.0956
Epoch 60, loss: 0.0875
Epoch 80, loss: 0.0839
Epoch 100, loss: 0.0918
Epoch 120, loss: 0.0812


In [ ]:
# =============================================================================
# GENERATE TEXT — sample characters one-by-one from the trained MiniGPT
# =============================================================================
# Starts with a 1-char prompt "F", then repeatedly:
#   look at last ≤ block_size chars → score next char → sample → append
#
# WHY OUTPUT LOOKED "INCOMPLETE" (cut off at "You "):
#   max_new_tokens=100 means ONLY 100 NEW characters after the prompt.
#   Total length ≈ 1 + 100 = 101 chars — enough to replay the start of the
#   memorized corpus, then the loop STOPS mid-sentence. Not a crash; budget done.
#
#   "First Citizen: ... First Citizen:\nYou "  ← ~101 chars, then halt.
#   Next would have been "are all resolved..." if you allow more tokens.
#
# Tiny corpus + low loss → model often REGURGITATES training text (overfit).
# temperature: lower ≈ greedier/more deterministic; higher ≈ more random.
#

context = torch.tensor([[char_to_idx['F']]], dtype=torch.long).to(device)
# Use a larger budget if you want the full memorized passage / longer samples
output_ids = model.generate(context, max_new_tokens=300, temperature=0.8)[0].tolist()
generated = ''.join([idx_to_char[i] for i in output_ids])
print(generated)
print(f"\n(length={len(generated)} chars = 1 prompt + {len(generated)-1} generated)")

# Now you have a tiny GPT. The architecture is identical to GPT‑2, just smaller. To scale up, you'd increase 
# embed_dim, num_layers, num_heads, and train on terabytes of text. That's exactly what the assistant's brain 
# will be, fine‑tuned to understand sales calls and whisper corrections.


First, you know Caius Marcius is chief enemy to the people.

Firstheak.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

Firstheay fpl:
tou know Caius Marcius is chief enemy to the people.

Firstheak.

First Citizen:
First, you know Caius Marcius is chief enemy to the peop

(length=301 chars = 1 prompt + 300 generated)


In [ ]:
# =============================================================================
# RECAP — what we built (step by step, with sticky examples)
# =============================================================================
# One-sentence goal:
#   A tiny ChatGPT-style brain that reads characters so far and guesses the NEXT one.
#
# ---------------------------------------------------------------------------
# STEP 0 — Big picture (vs notebook 6)
# ---------------------------------------------------------------------------
#   Notebook 6: encoder reads source, decoder writes answer (2 towers).
#   This notebook: ONE tower only (GPT / decoder-only).
#     prompt chars → model → next char → glue on → repeat.
#
#   Sticky example:
#     so far: "Befor"
#     guess:  "e"     → now "Before"
#
# ---------------------------------------------------------------------------
# STEP 1 — Corpus + character vocab
# ---------------------------------------------------------------------------
#   Raw Shakespeare-ish text → unique characters → integer IDs.
#     "B" → 8,  "e" → 14,  " " → 1   (IDs depend on sorted unique set)
#   Sticky: the model never sees letters; it sees numbers, then we map back.
#
# ---------------------------------------------------------------------------
# STEP 2 — Training windows (block_size = 32)
# ---------------------------------------------------------------------------
#   Cut overlapping chunks. X = context, y = same chunk shifted by 1.
#     X: B e f o r e _ w
#     y: e f o r e _ w e
#   At each seat: "given what I see here, predict the next char."
#   Sticky: block_size=32 → one look sees at most 32 characters (context window).
#
# ---------------------------------------------------------------------------
# STEP 3 — Causal self-attention + Transformer block
# ---------------------------------------------------------------------------
#   Multi-head attention, but MASK the future (lower triangle only):
#            B  e  f  o
#         B  ■  ·  ·  ·
#         e  ■  ■  ·  ·
#         f  ■  ■  ■  ·     · = cannot peek ahead
#         o  ■  ■  ■  ■
#   Block recipe (GPT-2 Pre-LN):
#     x = x + attn(LN(x))     # mix info from the PAST, keep residual sidewalk
#     x = x + mlp(LN(x))      # rewrite features per seat
#
# ---------------------------------------------------------------------------
# STEP 4 — MiniGPT full model
# ---------------------------------------------------------------------------
#   char ids → token embed ("what") + pos embed ("where")
#            → stack of blocks → LayerNorm → linear head → logits over vocab
#   Sticky: logits[t] = "scores for every possible next character at seat t."
#
# ---------------------------------------------------------------------------
# STEP 5 — Train (AdamW + cross-entropy)
# ---------------------------------------------------------------------------
#   Compare logits to true next chars in y. Loss down ⇒ better guesses.
#   Your run: ~3.0 → ~0.08  (tiny text ⇒ almost memorized — expected).
#   Sticky: autograd did the backprop we skipped writing by hand in NumPy.
#
# ---------------------------------------------------------------------------
# STEP 6 — Generate
# ---------------------------------------------------------------------------
#   Start with "F", sample next char, append, repeat max_new_tokens times.
#   Sticky: if it stopped at "You ", that was the TOKEN BUDGET (e.g. 100),
#           not the model "giving up." Raise max_new_tokens to continue.
#
# ---------------------------------------------------------------------------
# MEMORY CARD (fold this into your head)
# ---------------------------------------------------------------------------
#   tokenize chars → windows (X→y) → causal GPT blocks → CE train → sample.
#   Causal = look left only. Residual = keep original + edit. LN = steady volume.
#   Same skeleton as GPT-2 / ChatGPT — just character-level and tiny.
#
print("Recap cell only — scroll up for the working MiniGPT pipeline.")
